# Script 1 and 2 - Search Strategies Comparison

#### Objective
This notebook indexes CV data into a qdrant vector database using 3 different search strategies stored in separate in-memory collections:
1. **BM25**
2. **Vector Embedding**
3. **SPLADE**: It's an sparse model
3. **Hybrid with Reciprocal Rank fusion**
4. **Hybrid with Relative Score Fusion**

Search quality across all 4 strategies is evaluated against the ground truth matrix using Hit Rate, MRR@2, MRR@3, Precision, Recall, and NDCG@10.

### 1. Initializing a local client and creating the collections

Initializing Qdrant in  mode for fast testing.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating the collections for the different search strategies

In [2]:
# DENSE_MODEL = {
#     "size": 1024,
#     "model_name": "microsoft/harrier-oss-v1-0.6b",
# }

# Harrier is too heavy, so testing with a simple one
DENSE_MODEL = {
    "size": 384,
    "model_name": "avsolatorio/GIST-all-MiniLM-L6-v2",
}
BM25_MODEL = "Qdrant/bm25"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"
COLLECTION_NAME = "search_strategies"

client.create_collection(
    collection_name=COLLECTION_NAME,
    # Defining semantic search strategies
    vectors_config={
        "dense": models.VectorParams(
            size=DENSE_MODEL["size"],
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        # BM25 and lexical search strategies use IDF (Inverse Document Frequency). 
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
        # Neural sparse embedding models (e.g. SPLADE) do not need IDF
        "sparse": models.SparseVectorParams()
    }
)

True

### 2. Loading CV Data and embedding

In [3]:
import os
import json

# Load CV data
with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

Getting dense vector (Vector embedding)

In [4]:
from sentence_transformers import SentenceTransformer, SparseEncoder
from dotenv import load_dotenv

load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

dense_model = SentenceTransformer(DENSE_MODEL["model_name"], token=HUGGING_FACE_API_KEY)
sparse_model = SparseEncoder("naver/splade-v3")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 63915.59it/s]


In [5]:
from chonkie import RecursiveChunker

chunker = RecursiveChunker(
    tokenizer=sparse_model.tokenizer, 
    chunk_size=400, # splade is based on BERT, which has 512 tokens max
)

data_points = []

def is_less_than_max_tokens(ref_model, text, applicant_name, chunk_num):
    num_tokens = len(ref_model.tokenizer.encode(text))
    max_tokens = ref_model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

def serialize_cv_for_embedding(cv: dict) -> str:
    """Formats CV text specifically for optimal embedding model vectorization."""
    certifications = ", ".join(cv.get("certifications", [])) or "None"
    skills = ", ".join(cv.get("skills", [])) or "None"
    languages = ", ".join(cv.get("languages", [])) or "None"
    
    experience = "\n".join([f"- {exp}" for exp in cv.get("experience", [])])
    education = "\n".join([f"- {edu}" for edu in cv.get("education", [])])

    text = f"""Candidate Name: {cv.get("name")}
Profession: {cv.get("profession")}
Seniority Level: {cv.get("seniority_level")}
Location: {cv.get("location")}
Years of Experience: {cv.get("experience_years")}
Languages: {languages}

About Me:
{cv.get("about_me", "")}

Skills:
{skills}

Certifications:
{certifications}

Work Experience:
{experience}

Education:
{education}
""".strip()
    
    return text

for cv in cv_data:
    cv_text = serialize_cv_for_embedding(cv)
    chunks = chunker(cv_text)
    for c in chunks:
        chunk_str = c.text if hasattr(c, "text") else str(c)
        data_points.append({
            "chunk": chunk_str,
            "cv": cv
        })

chunks = [point["chunk"] for point in data_points]
dense_embeddings = dense_model.encode(chunks).tolist()
sparse_embeddings = sparse_model.encode_document(
    chunks,
    convert_to_sparse_tensor=True # Tensor with no zeroes
).coalesce()



Transforming the sparse embedding into a Qdrant format

In [6]:
doc_indices = sparse_embeddings.indices()[0].tolist()   # Which chunk?
token_indices = sparse_embeddings.indices()[1].tolist() # Which token?
weights = sparse_embeddings.values().tolist()           # The SPLADE score

qdrant_sparse_vectors = [
    {"indices": [], "values": []} for _ in range(len(chunks))
]

for doc_id, token_id, weight in zip(doc_indices, token_indices, weights):
    qdrant_sparse_vectors[doc_id]["indices"].append(token_id)
    qdrant_sparse_vectors[doc_id]["values"].append(weight)

sparse_embeddings = [
    models.SparseVector(indices=vec["indices"], values=vec["values"])
    for vec in qdrant_sparse_vectors
]

### 3. Uploading data points to Qdrant Collections

In [7]:
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            vector={
                "dense": dense_embeddings[idx],
                "bm25": models.Document(text=data_point["chunk"], model=BM25_MODEL, options={}),
                "sparse": sparse_embeddings[idx]
            
            },
            payload={
                **data_point["cv"],
                "chunk": data_point["chunk"]
            }
        )
        for idx, data_point in enumerate(data_points)
    ]
)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 13.37it/s]
